# GeoGuard: Satellite Landslide Detection
## Notebook 02: Preprocessing, Normalization & Data Preparation

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_loader import LandslideDatasetLoader
from src.preprocessing import SatellitePreprocessor

raw_dir = os.path.join(PROJECT_ROOT, 'data', 'raw')
loader = LandslideDatasetLoader(data_dir=raw_dir, pixel_threshold=100)
X_train, y_train, meta_train = loader.load_split('train')
X_val, y_val, meta_val = loader.load_split('validation')
print(f"X_train shape: {X_train.shape}, dtype: {X_train.dtype}")
print(f"X_val shape: {X_val.shape}, dtype: {X_val.dtype}")

### 2. Computing Training Statistics (Zero Data Leakage)

In [ ]:
preprocessor = SatellitePreprocessor(band_names=['B2', 'B3', 'B4', 'B8'])
preprocessor.fit(X_train)
for i, bname in enumerate(preprocessor.band_names):
    print(f"Band {bname:2s}: Mean = {preprocessor.mean[i]:.4f}, Std = {preprocessor.std[i]:.4f}")
X_train_norm = preprocessor.transform(X_train)
X_val_norm = preprocessor.transform(X_val)
print(f"Normalized train min/max: {X_train_norm.min():.2f} / {X_train_norm.max():.2f}")

### 3. Comparing Channel Distributions Before & After Normalization

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
band_names = ['B2 (Blue)', 'B3 (Green)', 'B4 (Red)', 'B8 (NIR)']
for c in range(4):
    axes[0, c].hist(X_train[:, :, :, c].flatten(), bins=40, color='royalblue', alpha=0.7)
    axes[0, c].set_title(f"Raw {band_names[c]}")
    axes[1, c].hist(X_train_norm[:, :, :, c].flatten(), bins=40, color='forestgreen', alpha=0.7)
    axes[1, c].set_title(f"Normalized {band_names[c]}")
plt.tight_layout()
plt.show()

### 4. Data Augmentations (Rotation and Flip Invariance)

In [ ]:
sample_patch = X_train[0]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(SatellitePreprocessor.to_rgb_display(sample_patch))
axes[0].set_title("Original Patch"); axes[0].axis('off')
for i in range(1, 4):
    aug = SatellitePreprocessor.augment_patch(sample_patch, seed=i*42)
    axes[i].imshow(SatellitePreprocessor.to_rgb_display(aug))
    axes[i].set_title(f"Augmented Variant #{i}"); axes[i].axis('off')
plt.tight_layout()
plt.show()

### 5. Saving Preprocessing Metadata for Model Serving

In [ ]:
config_path = os.path.join(PROJECT_ROOT, 'models', 'preprocessing_config.json')
preprocessor.save_config(config_path)
print(f"Saved config to {config_path}")